In [21]:
import sqlite3
import shutil
import pandas as pd

# Define database paths
original_db_path = "../Datasets/database/mfa.db"
copied_db_path = "Outputs/mfa.db"

# Make a copy of the original database
shutil.copy(original_db_path, copied_db_path)
print("Database copied to 'Outputs/mfa.db'.")

# Connect to the copied database
conn = sqlite3.connect(copied_db_path)
cursor = conn.cursor()

Database copied to 'Outputs/mfa.db'.


In [22]:
cursor.execute("SELECT sql FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

for table in tables:
    print(table[0])  # Print SQL CREATE statements

CREATE TABLE collections (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    accession_number TEXT NOT NULL UNIQUE,
    acquired NUMERIC
)
CREATE TABLE artists (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
)
CREATE TABLE created (
    artist_id INTEGER,
    collection_id INTEGER,
    PRIMARY KEY(artist_id, collection_id),
    FOREIGN KEY(artist_id) REFERENCES artists(id),
    FOREIGN KEY(collection_id) REFERENCES collections(id)
)


In [23]:
# Step 3: Function to execute a query and display results
def execute_and_display(query, conn):
    try:
        cursor.execute(query)
        conn.commit()
        print("Query executed successfully.")
    except sqlite3.IntegrityError as e:
        print(f"IntegrityError: {e}")
    except sqlite3.OperationalError as e:
        print(f"OperationalError: {e}")

# Step 4: Function to fetch and display a table
def fetch_table(table_name, conn):
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    print(df)

# Inserting

In [3]:
# Step 5: Insert individual rows
query1 = """
INSERT INTO collections (id, title, accession_number, acquired)
VALUES (5, 'test', '77.257', '1956-04-12');
"""
execute_and_display(query1, conn)

Query executed successfully.


In [5]:
table_name="collections"
fetch_table(table_name, conn)

   id                    title accession_number    acquired
0   1  Farmers working at dawn          11.6152  1911-08-03
1   2    Imaginative landscape           56.496        None
2   3     Profusion of flowers           56.257  1956-04-12
3   4            Spring outing            14.76  1914-01-08
4   5                     test           77.257  1956-04-12


In [12]:
# Adds a new item to the collections, demonstrating primary key auto-increments

query3 = """
INSERT INTO collections (title, accession_number, acquired)
VALUES ('Spring outing', '1466.76', '1914-01-08');
"""
execute_and_display(query3, conn)
fetch_table(table_name, conn)


Query executed successfully.
   id                    title accession_number    acquired
0   1  Farmers working at dawn          11.6152  1911-08-03
1   2    Imaginative landscape           56.496        None
2   3     Profusion of flowers           56.257  1956-04-12
3   4            Spring outing            14.76  1914-01-08
4   5                     test           77.257  1956-04-12
5   6            Spring outing          1466.76  1914-01-08


In [16]:
# This should cause a UNIQUE constraint violation
query4 = """
INSERT INTO collections (title, accession_number, acquired)
VALUES ('Spring outing', '14.76', '1914-01-08');  
"""
execute_and_display(query4, conn)
# fetch_table(table_name, conn)

IntegrityError: UNIQUE constraint failed: collections.accession_number
   id                    title accession_number    acquired
0   1  Farmers working at dawn          11.6152  1911-08-03
1   2    Imaginative landscape           56.496        None
2   3     Profusion of flowers           56.257  1956-04-12
3   4            Spring outing            14.76  1914-01-08
4   5                     test           77.257  1956-04-12
5   6            Spring outing          1466.76  1914-01-08


In [17]:
# -- This should cause a NOT NULL constraint violation
query5 = """
INSERT INTO collections (title, accession_number, acquired)
VALUES (NULL, '56.496', '1914-01-08');  
"""
execute_and_display(query5, conn)
# fetch_table(table_name, conn)

IntegrityError: NOT NULL constraint failed: collections.title
   id                    title accession_number    acquired
0   1  Farmers working at dawn          11.6152  1911-08-03
1   2    Imaginative landscape           56.496        None
2   3     Profusion of flowers           56.257  1956-04-12
3   4            Spring outing            14.76  1914-01-08
4   5                     test           77.257  1956-04-12
5   6            Spring outing          1466.76  1914-01-08


In [18]:
# Step 6: Insert multiple rows
multi_insert_query = """
INSERT INTO collections (title, accession_number, acquired) 
VALUES 
('Imaginative landscape', '56.1', NULL),
('Peonies and butterfly', '026.1899', '1906-01-01');
"""
execute_and_display(multi_insert_query, conn)
fetch_table(table_name, conn)

Query executed successfully.
   id                    title accession_number    acquired
0   1  Farmers working at dawn          11.6152  1911-08-03
1   2    Imaginative landscape           56.496        None
2   3     Profusion of flowers           56.257  1956-04-12
3   4            Spring outing            14.76  1914-01-08
4   5                     test           77.257  1956-04-12
5   6            Spring outing          1466.76  1914-01-08
6   7    Imaginative landscape             56.1        None
7   8    Peonies and butterfly         026.1899  1906-01-01


In [19]:
# Step 7: Display the collections table
fetch_table("collections", conn)

# Step 8: Close the connection
conn.close()


   id                    title accession_number    acquired
0   1  Farmers working at dawn          11.6152  1911-08-03
1   2    Imaginative landscape           56.496        None
2   3     Profusion of flowers           56.257  1956-04-12
3   4            Spring outing            14.76  1914-01-08
4   5                     test           77.257  1956-04-12
5   6            Spring outing          1466.76  1914-01-08
6   7    Imaginative landscape             56.1        None
7   8    Peonies and butterfly         026.1899  1906-01-01


# Updating


In [ ]:
# Updating authorship (incorrectly)
execute_and_display("""
UPDATE created SET artist_id = (
    SELECT id FROM artists WHERE name = 'Li Yin'
);
""", conn)
fetch_table("collections", conn)


In [ ]:
# Updating authorship (correctly)
execute_and_display("""
UPDATE created SET artist_id = (
    SELECT id FROM artists WHERE name = 'Li Yin'
)
WHERE collection_id = (
    SELECT id FROM collections WHERE title = 'Farmers working at dawn'
);
""", conn)
fetch_table("collections", conn)


In [ ]:
# Creating and inserting sample votes table
execute_and_display("""
CREATE TABLE IF NOT EXISTS votes (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL
);
""", conn)

# Insert sample votes
# Load votes from CSV
votes_df = pd.read_csv("Outputs/votes.csv")
for _, row in votes_df.iterrows():
    execute_and_display(f"INSERT INTO votes (title) VALUES ('{row['title']}');", conn)


In [ ]:

# Cleaning votes table
execute_and_display("UPDATE votes SET title = TRIM(title);", conn)
fetch_table("votes", conn)
execute_and_display("UPDATE votes SET title = UPPER(title);", conn)
fetch_table("votes", conn)

execute_and_display("UPDATE votes SET title = 'FARMERS WORKING AT DAWN' WHERE title LIKE 'Fa%';", conn)
fetch_table("votes", conn)

execute_and_display("UPDATE votes SET title = 'IMAGINATIVE LANDSCAPE' WHERE title LIKE 'Imag%';", conn)
fetch_table("votes", conn)

# Display votes table
fetch_table("votes", conn)

# Close the connection
conn.close()


In [ ]:
-- Demonstrates updating authorship
-- Uses mfa.db

-- Updates authorship (incorrectly)
UPDATE "created" SET "artist_id" = (
    SELECT "id" FROM "artists"
    WHERE "name" = 'Li Yin'
);

-- Updates authorship (correctly) for a piece with a previously unknown authorship
UPDATE "created" SET "artist_id" = (
    SELECT "id" FROM "artists"
    WHERE "name" = 'Li Yin'
)
WHERE "collection_id" = (
    SELECT "id" FROM "collections"
    WHERE "title" = 'Farmers working at dawn'
);


-- Demonstrates cleaning data from a CSV of votes for favorite artwork
-- Creates votes.db

-- Imports votes.csv
.import votes.csv votes

-- Counts votes
SELECT "title", COUNT("title") FROM "votes" GROUP BY "title";

-- Removes trailing whitespace
UPDATE "votes" SET "title" = trim("title");

-- Forces to uppercase
UPDATE "votes" SET "title" = upper("title");

-- Manually updates the titles of "Farmers working at dawn"
UPDATE "votes" SET "title" = 'FARMERS WORKING AT DAWN'
WHERE "title" = 'FARMERS WORKING';

UPDATE "votes" SET "title" = 'FARMERS WORKING AT DAWN'
WHERE "title" = 'FAMERS WORKING AT DAWN';

-- Fixes misspellings of "Farmers working at dawn"
UPDATE "votes" SET "title" = 'FARMERS WORKING AT DAWN'
WHERE "title" LIKE 'Fa%';

-- Fixes misspellings of "Imaginative landscape"
UPDATE "votes" SET "title" = 'IMAGINATIVE LANDSCAPE'
WHERE "title" LIKE 'Imag%';

-- Fixes misspellings of "Profusion of flowers"
UPDATE "votes" SET "title" = 'PROFUSION OF FLOWERS'
WHERE "title" LIKE 'Profusion %';


In [ ]:
# Deleting

-- Demonstrates deleting rows from a single table
-- Uses mfa.db

-- Deletes item with particular title
DELETE FROM "collections" WHERE "title" = 'Spring outing';

-- Deletes item where value is NULL
DELETE FROM "collections" WHERE "acquired" IS NULL;

-- Deletes items acquired before the museum moved to a new location in 1909
DELETE FROM "collections" WHERE "acquired" < '1909-01-01';

-- Demonstrates deleting rows with constraints
-- Uses mfa.db

-- Raises a foreign key constraint error
DELETE FROM "artists" WHERE "name" = 'Unidentified artist';

-- Deletes the artist's affiliation with their work, using hard-coded id
DELETE FROM "created" WHERE "artist_id" = 3;

-- Deletes the artist's affiliation with their work, using subquery
DELETE FROM "created" WHERE "artist_id" = (
    SELECT "id" FROM "artists" WHERE "name" = 'Unidentified artist'
);

-- Deletes the artist themselves
DELETE FROM "artists" WHERE "name" = 'Unidentified artist';


In [ ]:
# Triggers

-- Demonstrates triggers on delete and insert
-- Uses mfa.db

-- Creates a table to track buying and selling of items from collections
CREATE TABLE "transactions" (
    "id" INTEGER,
    "title" TEXT,
    "action" TEXT,
    PRIMARY KEY("id")
);

-- Creates a trigger to log selling items from collections
CREATE TRIGGER "sell" 
BEFORE DELETE ON "collections"
BEGIN
    INSERT INTO "transactions" ("title", "action")
    VALUES (OLD."title", 'sold');
END;

-- Lists existing triggers
.schema

-- Deletes from collections
DELETE FROM "collections" WHERE "title" = 'Profusion of flowers';

-- Creates a trigger to log buying items
CREATE TRIGGER "buy" 
AFTER INSERT ON "collections"
BEGIN
    INSERT INTO "transactions" ("title", "action")
    VALUES (NEW."title", 'bought');
END;

-- Adds item to collections
INSERT INTO "collections" ("title", "accession_number", "acquired")
VALUES ('Profusion of flowers', '56.257', '1956-04-12');


In [ ]:
# Soft delete



-- Demonstrates soft deletes
-- Uses mfa.db

-- Adds a "deleted" column to "collections" table
ALTER TABLE "collections" ADD COLUMN "deleted" INTEGER DEFAULT 0;

-- Views updated schema of collections table
.schema "collections"

-- Views data
SELECT * FROM "collections";

-- Instead of deleting an item, updates its deleted column to be 1
UPDATE "collections" SET "deleted" = 1 WHERE "title" = 'Farmers working at dawn';

-- Selects all items from collections that are not deleted
SELECT * FROM "collections" WHERE "deleted" != 1;



-- Creates a view to show only items in collections that are NOT deleted
CREATE VIEW "current_collections" AS
SELECT "id", "title", "accession_number", "acquired" FROM "collections" WHERE "deleted" = 0;

-- Selects from "current_collections" view to see non-deleted items
SELECT * FROM "current_collections";

-- Fails to delete an item from the view
DELETE FROM "current_collections" WHERE "title" = 'Imaginative landscape';

-- Creates trigger to delete items from a view
CREATE TRIGGER "delete"
INSTEAD OF DELETE ON "current_collections"
FOR EACH ROW
BEGIN
    UPDATE "collections" SET "deleted" = 1 WHERE "id" = OLD."id";
END;

-- Creates trigger to revert an item's deletion
CREATE TRIGGER "insert_when_exists"
INSTEAD OF INSERT ON "current_collections"
FOR EACH ROW 
WHEN NEW."accession_number" IN (SELECT "accession_number" FROM "collections")
BEGIN
    UPDATE "collections" SET "deleted" = 0 WHERE "accession_number" = NEW."accession_number";
END;

-- Creates trigger to insert a new item into collections
CREATE TRIGGER "insert_when_new"
INSTEAD OF INSERT ON "current_collections"
FOR EACH ROW
WHEN NEW."accession_number" NOT IN (SELECT "accession_number" FROM "collections")
BEGIN
    INSERT INTO "collections" ("title", "accession_number", "acquired")
    VALUES (NEW."title", NEW."accession_number", NEW."acquired");
END;
